# Trying to Get Bluesky with Model 3

## References
1. Bluesky Tutorial for [Ophyd Setup of Single EPICS PV](https://blueskyproject.io/tutorials/Single%20PVs.html)
2. Bluesky Tutorial on [Reading from Detectors](https://blueskyproject.io/tutorials/Hello%20Bluesky.html)

## Configure a Single EPICS PV using Ophyd

I'll start with something simple: reading the trigger count. (From the Pulse Generator)

In [1]:
from ophyd.signal import EpicsSignal, EpicsSignalRO

trig_count = EpicsSignalRO("PULSEGEN:trigger:count", name="trig_count")

In [2]:
trig_count.wait_for_connection()

## Print the value of the EPICS PV

In [3]:
trig_count.get()

126846.0

## Make a BlueSky Chart of the EPICS PV

Start the Bluesky Run Engine

In [4]:
from bluesky import RunEngine
from bluesky.callbacks import LiveTable
from bluesky.plans import count, list_scan

RE = RunEngine()

In [5]:
token = RE.subscribe(LiveTable(["trig_count"]))

RE(count([trig_count], num=5, delay=0.5))  # Read every 0.5 seconds.



+-----------+------------+------------+
|   seq_num |       time | trig_count |
+-----------+------------+------------+
|         1 | 11:42:51.7 |     126996 |
|         2 | 11:42:52.2 |     127046 |
|         3 | 11:42:52.7 |     127046 |
|         4 | 11:42:53.2 |     127096 |
|         5 | 11:42:53.7 |     127096 |
+-----------+------------+------------+
generator count ['448bff84'] (scan num: 1)




('448bff84-14e7-4ca9-8058-7bfc70c5a3cd',)

That's not ideal necessarily, since we have readings every 0.5 seconds whether anything has been updated in our system. The PVs are out of date since our PV really only updates every so often maybe once per second?

## Use CAMonitor instead

In [8]:
from collections import deque
import time

def accumulate(value, old_value, timestamp, **kwargs):
   readings.append({"trig_count": {"value": value, "timestamp": timestamp}})

readings = deque(maxlen=5)
trig_count.unsubscribe_all() # clear any old subscriptions to avoid duplicate printing (if you had old ones)
trig_count.subscribe(accumulate)

1

In [9]:
time.sleep(3)
readings

deque([{'trig_count': {'value': 129046.0, 'timestamp': 1781808211.830331}},
       {'trig_count': {'value': 129096.0, 'timestamp': 1781808212.831949}},
       {'trig_count': {'value': 129146.0, 'timestamp': 1781808213.830467}},
       {'trig_count': {'value': 129196.0, 'timestamp': 1781808214.830888}},
       {'trig_count': {'value': 129246.0, 'timestamp': 1781808215.830049}}],
      maxlen=5)

That's better!